# Quickstart: Ramp-Event Forecasting Ensemble

**Paper:** Stergiou & Karakasidis (2026), Energies 
**GitHub:** https://github.com/USERNAME/REPO

This notebook demonstrates the full training and evaluation pipeline on **synthetic data** 
so you can verify the code runs correctly without needing the SCADA files.

Runtime: CPU is sufficient for this demo (synthetic, small dataset).  
For full training on real data: `Runtime → Change runtime type → A100 GPU`.

Important Note: These synthetic-demo numbers are not the paper's results — they only confirm the pipeline runs end to end. The paper's reported metrics (76.5% ramp-DA, etc.) come from training on real SCADA data via wind_pipeline.py

In [ ]:
# Install dependencies
!pip install torch numpy scipy matplotlib tqdm -q

# Clone repo (or upload manually)
import os
if not os.path.exists('ramp-event-forecasting'):
    !git clone https://github.com/USERNAME/REPO.git
%cd ramp-event-forecasting

import sys
sys.path.insert(0, '.')
print('Setup complete.')

In [ ]:
# ── Generate synthetic wind power data ──────────────────────────────
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

N_SAMPLES  = 5000
INPUT_DIM  = 20
LOOKBACK   = 48
FORECAST   = 6
RAMP_THR   = 0.05

# Synthetic features: diurnal power + noise
t     = np.arange(N_SAMPLES) / 144   # days
power = 0.4 + 0.3*np.sin(2*np.pi*t) + 0.1*np.random.randn(N_SAMPLES)
power = np.clip(power, 0, 1).astype(np.float32)

# Build 20-column feature matrix
feats = np.zeros((N_SAMPLES, INPUT_DIM), dtype=np.float32)
feats[:, 0]  = 0.5 + 0.2*np.sin(2*np.pi*t) + 0.05*np.random.randn(N_SAMPLES)  # ws
feats[:, 1]  = power                                                              # power (TARGET)
feats[:, 2]  = np.sin(2*np.pi*(np.arange(N_SAMPLES) % 144) / 144)              # hour_sin
feats[:, 3]  = np.cos(2*np.pi*(np.arange(N_SAMPLES) % 144) / 144)              # hour_cos
for j in range(4, INPUT_DIM):                                                    # lags / derived
    feats[:, j] = np.roll(feats[:, j % 4], j) + 0.01*np.random.randn(N_SAMPLES)
feats = np.clip(feats, 0, 1)

train_end = int(N_SAMPLES * 0.70)
val_end   = int(N_SAMPLES * 0.85)
print(f'Samples: {N_SAMPLES}  train={train_end}  val={val_end-train_end}  test={N_SAMPLES-val_end}')

# Ramp-event prevalence in synthetic data
delta = np.abs(np.diff(power, prepend=power[0]))
print(f'Ramp-event prevalence: {(delta > RAMP_THR).mean()*100:.1f}%')

In [ ]:
# ── Dataset & DataLoader ────────────────────────────────────────────
COL_POWER = 1

class SimpleDataset(Dataset):
    def __init__(self, feats, indices, lookback, forecast_steps):
        self.feats, self.lb, self.fs, self.indices = feats, lookback, forecast_steps, indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        j    = self.indices[i]
        x    = torch.from_numpy(self.feats[j-self.lb:j])
        y    = torch.from_numpy(self.feats[j:j+self.fs, COL_POWER])
        last = torch.tensor(self.feats[j-1, COL_POWER])
        return x, y, last

def make_idx(start, end, lb):
    return [i for i in range(lb+FORECAST, N_SAMPLES) if start <= i < end and i+FORECAST <= N_SAMPLES]

tr_idx = make_idx(0,         train_end, LOOKBACK)
vl_idx = make_idx(train_end, val_end,   LOOKBACK)
te_idx = make_idx(val_end,   N_SAMPLES, LOOKBACK)

tr_ds = SimpleDataset(feats, tr_idx, LOOKBACK, FORECAST)
vl_ds = SimpleDataset(feats, vl_idx, LOOKBACK, FORECAST)
te_ds = SimpleDataset(feats, te_idx, LOOKBACK, FORECAST)

tr_loader = DataLoader(tr_ds, batch_size=64, shuffle=True)
vl_loader = DataLoader(vl_ds, batch_size=64)
te_loader = DataLoader(te_ds, batch_size=64)
print(f'Datasets: train={len(tr_ds)}  val={len(vl_ds)}  test={len(te_ds)}')

In [ ]:
# ── Import specialist models and loss ───────────────────────────────
from src.models.specialists import DailyPatternSpecialist, WeatherFrontSpecialist
from src.losses.multiobjective import multi_objective_loss
from src.eval.metrics import compute_all_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Instantiate two specialists for the demo
daily   = DailyPatternSpecialist(LOOKBACK, FORECAST, INPUT_DIM, dropout=0.2).to(device)
weather = WeatherFrontSpecialist(LOOKBACK, FORECAST, INPUT_DIM, hidden=64, num_heads=4, 
                                  ff_dim=64, num_layers=2, dropout=0.1).to(device)

n_daily   = sum(p.numel() for p in daily.parameters())
n_weather = sum(p.numel() for p in weather.parameters())
print(f'DailyPatternSpecialist params:  {n_daily:,}')
print(f'WeatherFrontSpecialist params:  {n_weather:,}')

In [ ]:
# ── Quick training demo (100 epochs) ────────────────────────────────
EPOCHS   = 100
opt      = torch.optim.AdamW(daily.parameters(), lr=5e-4)
params   = {'alpha': 1.3, 'beta': 0.48, 'gamma': 0.036}

print('Training DailyPatternSpecialist (100 epochs demo)...')
for ep in range(1, EPOCHS+1):
    daily.train()
    tr_losses = []
    for x, y, last in tr_loader:
        x, y, last = x.to(device), y.to(device), last.to(device)
        opt.zero_grad()
        pred = daily(x)
        loss, _ = multi_objective_loss(pred, y, last, last, **params)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(daily.parameters(), 1.0)
        opt.step()
        tr_losses.append(loss.item())
    
    # Validation
    daily.eval()
    preds_v, trues_v, lasts_v = [], [], []
    with torch.no_grad():
        for x, y, last in vl_loader:
            p = daily(x.to(device)).cpu().numpy()[:, 0]
            preds_v.append(p); trues_v.append(y[:,0].numpy()); lasts_v.append(last.numpy())
    import numpy as np
    pv = np.concatenate(preds_v); tv = np.concatenate(trues_v); lv = np.concatenate(lasts_v)
    m  = compute_all_metrics(tv, pv, lv)
    print(f'  ep {ep:2d}  tr_loss={np.mean(tr_losses):.4f}  '
          f'val_DA={m["directional_accuracy"]:.4f}  '
          f'val_ramp_DA={m["ramp_directional_accuracy"]:.4f}  '
          f'MAE={m["mae"]:.4f}')

In [ ]:
# ── Evaluate on test set ─────────────────────────────────────────────
daily.eval()
preds_t, trues_t, lasts_t = [], [], []
with torch.no_grad():
    for x, y, last in te_loader:
        p = daily(x.to(device)).cpu().numpy()[:, 0]
        preds_t.append(p); trues_t.append(y[:,0].numpy()); lasts_t.append(last.numpy())

pt = np.concatenate(preds_t); tt = np.concatenate(trues_t); lt = np.concatenate(lasts_t)
m  = compute_all_metrics(tt, pt, lt)

print('\n=== Test set results (synthetic data, 10-epoch demo) ===')
print(f'  MAE                     : {m["mae"]:.4f}')
print(f'  RMSE                    : {m["rmse"]:.4f}')
print(f'  Pearson r               : {m["pearson_r"]:.4f}')
print(f'  Directional accuracy    : {m["directional_accuracy"]*100:.2f}%')
print(f'  Ramp-event DA           : {m["ramp_directional_accuracy"]*100:.2f}%')
print(f'  Stable-period DA        : {m["stable_directional_accuracy"]*100:.2f}%')
print(f'  Weighted DA             : {m["weighted_directional_accuracy"]*100:.2f}%')
print(f'  Turning-point accuracy  : {m["turning_point_accuracy"]*100:.2f}%')
print()
print('Pipeline verified. For real results, run wind_pipeline.py on your SCADA data.')